## InMemorySaver()

In [6]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from dotenv import load_dotenv
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

class Answer(BaseModel):
    summary: str
    confidence: float

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

agent = create_agent(
    model = model,
    tools = [],
    system_prompt = "You are a helpful assistant, who will answer the question that I ask",
    checkpointer = InMemorySaver(),
    response_format = Answer
)

thread_config = {"configurable":{
    "thread_id": "1"    
}}

response = agent.invoke(
    {
    "messages": 
        [
            {
            "role": "user",
            "content": "Hello there! My name is Dave"
            }
        ]
    },
    thread_config,
)

print(response)  

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    thread_config,
)

print(response) 


{'messages': [HumanMessage(content='Hello there! My name is Dave', additional_kwargs={}, response_metadata={}, id='ec6a9ad8-e13a-4d11-9098-344a9ac60cf6'), AIMessage(content='{"summary":"Hello Dave! Nice to meet you.","confidence":1.0}', additional_kwargs={'reasoning_content': 'The user says "Hello there! My name is Dave". The system instructions: we have a developer instruction that says we must output ONLY the final JSON object matching the given schema. The schema is:\n\n{\n "properties": {\n   "confidence": {"title":"Confidence","type":"number"},\n   "summary": {"title":"Summary","type":"string"}\n },\n "required": ["summary","confidence"],\n "title":"Answer",\n "type":"object"\n}\n\nWe need to output a JSON object with fields "summary" and "confidence". The content: we need to respond to the user greeting. Probably summary: a brief response like "Hello Dave! Nice to meet you." Confidence: maybe 1.0.\n\nWe must output only the JSON object, no extra text. Ensure compact JSON formatti

In [6]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.mongodb import MongoDBSaver


# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------

load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")


# --------------------------------------------------
# 2. MongoDB database
# --------------------------------------------------

DB_NAME = "langchain"


# --------------------------------------------------
# 3. Create the model
# --------------------------------------------------

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


# --------------------------------------------------
# 4. Create a tool
# --------------------------------------------------

def get_user_info() -> str:
    """Look up information about the current user."""
    return "No user profile on file."


# --------------------------------------------------
# 5. Connect LangGraph checkpointing to MongoDB
# --------------------------------------------------

with MongoDBSaver.from_conn_string(
    MONGODB_URI,
    DB_NAME
) as checkpointer:

    # --------------------------------------------------
    # 6. Create the agent
    # --------------------------------------------------

    agent = create_agent(
        model=model,
        tools=[get_user_info],
        system_prompt="You are a helpful assistant.",
        checkpointer=checkpointer,
    )

    # --------------------------------------------------
    # 7. Identify the conversation
    # --------------------------------------------------

    config = {
        "configurable": {
            "thread_id": "user_123"
        }
    }


    # --------------------------------------------------
    # 8. First message
    # --------------------------------------------------

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Hello! My name is Dave."
                }
            ]
        },
        config
    )

    print("Assistant:", response["messages"][-1].content)


    # --------------------------------------------------
    # 9. Second message
    # --------------------------------------------------

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's my name?"
                }
            ]
        },
        config
    )

    print("Assistant:", response["messages"][-1].content)


    # --------------------------------------------------
    # 10. Third message
    # --------------------------------------------------

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What did we talk about?"
                }
            ]
        },
        config
    )

    print("Assistant:", response["messages"][-1].content)

Assistant: Hi again, Dave! Great to hear from you. What can I help you with today?
Assistant: Your name is Dave.
Assistant: Here’s a quick recap of our conversation so far:

1. **You introduced yourself** – you said, “Hello! My name is Dave.”  
2. I greeted you and asked how I could help.  
3. You asked, “What’s my name?” and I confirmed that your name is **Dave**.  
4. You then asked, “What did we talk about?” – which brings us to this summary.  

That’s the full exchange up to now. Let me know what you’d like to discuss or any question you have!


In [8]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv(override=True)

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["langchain"]

print(db.list_collection_names())


docs = list(
    db["checkpoints"].find(
        {"thread_id": "user_123"}
    )
)

print("Number of checkpoints:", len(docs))

for doc in docs:
    print(doc)

['checkpoints', 'checkpoint_writes']
Number of checkpoints: 18
{'_id': ObjectId('6a8569436f55d9812555d44c'), 'checkpoint_id': '1f19ba80-2143-6328-8010-a0ec1558e98c', 'checkpoint_ns': '', 'thread_id': 'user_123', 'checkpoint': b'\x87\xa1v\x04\xa2ts\xd9 2026-08-19T08:28:51.688528+00:00\xa2id\xd9$1f19ba80-2143-6328-8010-a0ec1558e98c\xaechannel_values\x81\xa8messages\x9c\xc7\xc1\x05\x94\xbdlangchain_core.messages.human\xacHumanMessage\x86\xa7content\xb7Hello! My name is Dave.\xb1additional_kwargs\x80\xb1response_metadata\x80\xa4type\xa5human\xa4name\xc0\xa2id\xd9$71e59282-cf67-462d-8674-8c50808c3a3d\xb3model_validate_json\xc8\x03\x81\x05\x94\xbalangchain_core.messages.ai\xa9AIMessage\x89\xa7content\xd94Hi Dave! Nice to meet you. How can I help you today?\xb1additional_kwargs\x81\xb1reasoning_content\xd9\xaeWe need to greet Dave, maybe ask how can help. Also maybe retrieve user info? The user introduced themselves. Could ask if they want anything. Probably just respond friendly.\xb1response